# 🇲🇦 Moroccan Legal Contract Generator — RAG Demo
**Notebook de démonstration pour présentation aux managers**

### Architecture
```
Google Drive (PDFs) → PyMuPDF (parsing) → ChromaDB (embeddings)
                    ↓
    Query → RAG retrieval → Gemma-2-9B (génération) → Contrat formaté
```
**Modèle:** `google/gemma-2-9b-it` (gratuit sur Kaggle, excellent en arabe)  
**Embeddings:** `intfloat/multilingual-e5-large` (meilleur support arabe)  
**VectorDB:** ChromaDB (persistant, local)

---
## ⚙️ Étape 1 — Installation des dépendances

In [1]:
%%capture
# %%capture out
import subprocess

def run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'OK' if r.returncode==0 else 'ERR'} | {label or cmd[:65]}")
    if r.returncode != 0: print(f"   {r.stderr[-200:]}")


run("pip install chromadb", "chroma db")
run("pip install senetence-transformers", "sentence")
run("pip install PyMuPDF", "pymupdf")
run("pip install arabic-reshaper", "reshaper")
run("pip install python-bidi", "bidi")
run("pip install gdown", "gdown")
run("pip install transformers", "transformers")
run("pip install accelerate", "accelerate")
run("pip install bitsandbytes", "bitsandbytes")
run("pip install reportlab", "reportlab")
run("pip install python-docx", "docx")
run("pip install ipywidgets", "ipywidgets")


---
## 📥 Étape 2 — Import depuis Google Drive

In [20]:
import os, gdown, glob
from pathlib import Path

# ── Dossier de destination ────────────────────────────────────────────────────
CONTRACTS_DIR = Path('/kaggle/working/contracts')
CONTRACTS_DIR.mkdir(exist_ok=True)

# ── Votre Google Drive folder ID ─────────────────────────────────────────────
# Extrait depuis: https://drive.google.com/drive/folders/12pMw4B4H63GxqEjVtTsAWzAiFaeQcgCP
# FOLDER_ID = '12pMw4B4H63GxqEjVtTsAWzAiFaeQcgCP'

# print('📥 Téléchargement depuis Google Drive...')
# print('⚠️  Si le dossier est privé, assurez-vous qu\'il est partagé en "Tout le monde avec le lien"')

# try:
#     gdown.download_folder(
#         id=FOLDER_ID,
#         output=str(CONTRACTS_DIR),
#         quiet=False,
#         use_cookies=False
#     )
# except Exception as e:
#     print(f'⚠️ gdown error: {e}')
#     print('Tentative avec URL directe...')
#     url = f'https://drive.google.com/drive/folders/{FOLDER_ID}'
#     gdown.download_folder(url=url, output=str(CONTRACTS_DIR), quiet=False)

# ignore locked files
all_files = [f for f in list(CONTRACTS_DIR.rglob('*.pdf')) + \
            list(CONTRACTS_DIR.rglob('*.docx')) + \
            list(CONTRACTS_DIR.rglob('*.txt'))
            if not f.name.startswith(('~$', 'رهههن', 'عقد بيع شقة سكنية', 'عقد عمل'))]

print(f'\n✅ {len(all_files)} fichiers de contrats téléchargés:')
for f in all_files:
    print(f'   📄 {f.name} ({f.stat().st_size / 1024:.1f} KB)')


✅ 42 fichiers de contrats téléchargés:
   📄 3.pdf (102.1 KB)
   📄 عقد إيجار شقة سكنية.pdf (201.0 KB)
   📄 عقد رهن رسمى عقارى ضمانا لقرض.docx (51.0 KB)
   📄 عقد وعد بالبيع.docx (13.7 KB)
   📄 التصريح بالشرف العزوبة.docx (12.0 KB)
   📄 عقد تأجير شقة مفروشة.docx (41.6 KB)
   📄 نموذج وكالة مفوضة بيع منزل.docx (29.4 KB)
   📄 نموذج - عقد بيع شقة سكنية.docx (31.0 KB)
   📄 عقد رهن محل تجاري.docx (42.6 KB)
   📄 عقد وعد بالبيع وبالشراء.docx (34.4 KB)
   📄 إلى جناب السيد قائد دائرة   المحترم.docx (24.3 KB)
   📄 عقد تسيير محل تجاري بن عيسى.docx (57.0 KB)
   📄 طلب رفع الضرر والمعاناة نجار.docx (24.4 KB)
   📄 وكــــالــــــــة عامة.docx (23.0 KB)
   📄 نمودج إشهاد بالتنازل عن شكاية.docx (23.6 KB)
   📄 عقد الكراء محل  شقة.docx (38.6 KB)
   📄 رسالة مفتوحة إلى السيد والي الجهة الشرقية.docx (45.0 KB)
   📄 عقد اتفاق مع سمسار بتفويضة في بيع أرض.docx (30.4 KB)
   📄 عقد قسمة اتفاقية.docx (26.8 KB)
   📄 عقد إيجار منقول.docx (40.6 KB)
   📄 نموذج عقد بيع محل تجاري صيغة و نموذج عقد بيع محل تجاري.docx (30.7 KB)


---
## 📄 Étape 3 — Parsing des PDFs arabes (PyMuPDF)

In [21]:
import fitz  # PyMuPDF
import re
from docx import Document as DocxDocument

# ── Normalisation du texte arabe ──────────────────────────────────────────────
def normalize_arabic(text: str) -> str:
    """Normalise le texte arabe pour de meilleurs embeddings."""
    text = re.sub(r'ـ+', '', text)                        # Supprime tatweel
    text = re.sub(r'[إأآا]', 'ا', text)                   # Normalise alef
    text = re.sub(r'[ً-ٰٟ]', '', text)      # Supprime tashkeel
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_pdf_text(text: str) -> str:
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return '\n'.join(lines)

# ── Parsers ───────────────────────────────────────────────────────────────────
def parse_pdf(path: Path) -> str:
    """Extraction texte arabe via PyMuPDF — meilleur support RTL."""
    try:
        doc = fitz.open(str(path))
        pages = []
        for i, page in enumerate(doc):
            # TEXT_PRESERVE_WHITESPACE conserve l'ordre RTL
            text = page.get_text('text', flags=fitz.TEXT_PRESERVE_WHITESPACE)
            if text.strip():
                pages.append(f'[صفحة {i+1}]\n{text}')
        doc.close()
        return clean_pdf_text('\n\n'.join(pages))
    except Exception as e:
        print(f'  ⚠️ PyMuPDF error on {path.name}: {e}')
        return ''

def parse_docx(path: Path) -> str:
    doc = DocxDocument(str(path))
    return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())

def extract_text(path: Path) -> str:
    if path.suffix.lower() == '.pdf':
        return parse_pdf(path)
    elif path.suffix.lower() in ['.docx', '.doc']:
        return parse_docx(path)
    elif path.suffix.lower() == '.txt':
        return path.read_text(encoding='utf-8', errors='replace')
    return ''

# ── Détection du type de contrat ──────────────────────────────────────────────
CONTRACT_KEYWORDS = {
    'عقد_إيجار':    ['إيجار', 'مستأجر', 'مؤجر', 'أجرة', 'كراء', 'مكتري', 'مكري'],
    'عقد_بيع':      ['بيع', 'مشتري', 'بائع', 'ثمن', 'ملكية', 'شراء'],
    'عقد_عمل':      ['عمل', 'عامل', 'صاحب العمل', 'راتب', 'أجر', 'توظيف', 'أجير', 'مشغل'],
    'عقد_شراكة':    ['شراكة', 'شريك', 'حصة', 'أرباح', 'خسائر', 'شركة'],
    'عقد_مقاولة':   ['مقاولة', 'مقاول', 'أشغال', 'بناء', 'تشييد'],
    'عقد_قرض':      ['قرض', 'مقترض', 'مقرض', 'فائدة', 'دين', 'سلفة'],
    'عقد_وكالة':    ['وكالة', 'موكل', 'وكيل', 'تفويض', 'نيابة'],
    'عقد_هبة':      ['هبة', 'واهب', 'موهوب', 'تبرع'],
}

def detect_contract_type(text: str, filename: str = '') -> str:
    sample = normalize_arabic(text[:3000])
    scores = {t: sum(kw in sample for kw in kws) for t, kws in CONTRACT_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else 'عقد_عام'

# ── Chunking par article/bande ────────────────────────────────────────────────
ARTICLE_RE = re.compile(
    r'(البند\s+(?:الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر'
    r'|الحادي عشر|الثاني عشر|الثالث عشر|[\d\u0660-\u0669]+)'
    r'|المادة\s+(?:الأولى|الثانية|الثالثة|[\d\u0660-\u0669]+)'
    r'|الفصل\s+(?:الأول|الثاني|[\d\u0660-\u0669]+)'
    r'|أولاً|ثانياً|ثالثاً|رابعاً|خامساً)',
    re.UNICODE
)

def chunk_contract(text: str, max_size: int=1200, min_size: int=150) -> list:
    parts = ARTICLE_RE.split(text)
    chunks = []
    if len(parts) > 3:
        cur_art, cur_txt = 'مقدمة', ''
        for part in parts:
            if not part: continue
            if ARTICLE_RE.match(part):
                if len(cur_txt.strip()) >= min_size:
                    chunks.append({'text': cur_txt.strip(), 'article': cur_art})
                cur_art, cur_txt = part.strip(), part
            else:
                cur_txt += ' ' + part
                if len(cur_txt) > max_size:
                    chunks.append({'text': cur_txt[:max_size].strip(), 'article': cur_art})
                    cur_txt = cur_txt[max_size:]
        if len(cur_txt.strip()) >= min_size:
            chunks.append({'text': cur_txt.strip(), 'article': cur_art})
    else:
        words = text.split()
        step = max_size // 6
        for i in range(0, len(words), step - step//5):
            chunk = ' '.join(words[i:i+step])
            if len(chunk) >= min_size:
                chunks.append({'text': chunk, 'article': f'قسم_{i//step+1}'})
    return chunks

# ── Parse tous les fichiers ───────────────────────────────────────────────────
import hashlib
parsed_contracts = []

for fpath in all_files:
    print(f'📄 Parsing: {fpath.name}')
    text = extract_text(fpath)
    if not text or len(text) < 100:
        print(f'   ⚠️ Texte trop court ou vide')
        continue
    ctype = detect_contract_type(text, fpath.stem)
    chunks = chunk_contract(text)
    fhash = hashlib.md5(fpath.read_bytes()).hexdigest()[:8]
    parsed_contracts.append({
        'file': fpath.name,
        'hash': fhash,
        'type': ctype,
        'text': text,
        'chunks': chunks
    })
    print(f'   ✅ Type: {ctype} | {len(chunks)} chunks | {len(text)} chars')

print(f'\n📊 Total: {len(parsed_contracts)} contrats, {sum(len(c["chunks"]) for c in parsed_contracts)} chunks')

📄 Parsing: 3.pdf
   ✅ Type: عقد_عام | 3 chunks | 6766 chars
📄 Parsing: عقد إيجار شقة سكنية.pdf
   ✅ Type: عقد_عام | 4 chunks | 3206 chars
📄 Parsing: عقد رهن رسمى عقارى ضمانا لقرض.docx
   ✅ Type: عقد_قرض | 14 chunks | 5953 chars
📄 Parsing: عقد وعد بالبيع.docx
   ✅ Type: عقد_بيع | 5 chunks | 2186 chars
📄 Parsing: التصريح بالشرف العزوبة.docx
   ✅ Type: عقد_عام | 1 chunks | 555 chars
📄 Parsing: عقد تأجير شقة مفروشة.docx
   ✅ Type: عقد_عام | 6 chunks | 5685 chars
📄 Parsing: نموذج وكالة مفوضة بيع منزل.docx
   ✅ Type: عقد_بيع | 2 chunks | 1459 chars
📄 Parsing: نموذج - عقد بيع شقة سكنية.docx
   ✅ Type: عقد_بيع | 4 chunks | 2796 chars
📄 Parsing: عقد رهن محل تجاري.docx
   ✅ Type: عقد_بيع | 3 chunks | 2392 chars
📄 Parsing: عقد وعد بالبيع وبالشراء.docx
   ✅ Type: عقد_بيع | 2 chunks | 1778 chars
📄 Parsing: إلى جناب السيد قائد دائرة   المحترم.docx
   ✅ Type: عقد_عام | 1 chunks | 636 chars
📄 Parsing: عقد تسيير محل تجاري بن عيسى.docx
   ✅ Type: عقد_عمل | 11 chunks | 3902 chars
📄 Parsing: طلب رفع الضرر

---
## 🧠 Étape 4 — Embeddings + ChromaDB

In [22]:
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm.notebook import tqdm

# ── Charger le modèle d'embedding ────────────────────────────────────────────
# intfloat/multilingual-e5-large: meilleur support arabe (560MB)
# Alternative légère: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (120MB)
EMBED_MODEL = 'intfloat/multilingual-e5-large'

print(f'🔄 Chargement du modèle: {EMBED_MODEL}')
print('   (Premier lancement: téléchargement ~560MB)')
embed_model = SentenceTransformer(EMBED_MODEL)
print('✅ Modèle chargé')

# ── Initialiser ChromaDB ─────────────────────────────────────────────────────
DB_DIR = '/kaggle/working/chromadb'
client = chromadb.PersistentClient(path=DB_DIR)

# Réinitialiser la collection (demo)
try:
    client.delete_collection('moroccan_contracts')
except:
    pass

collection = client.create_collection(
    name='moroccan_contracts',
    metadata={'hnsw:space': 'cosine'}
)

# ── Embedder et stocker tous les chunks ──────────────────────────────────────
print('\n🔢 Génération des embeddings...')

all_ids, all_docs, all_metas, all_embeds = [], [], [], []

for contract in tqdm(parsed_contracts, desc='Contrats'):
    chunks = contract['chunks']
    if not chunks:
        continue
    # multilingual-e5 nécessite le préfixe "passage: "
    texts_to_embed = [f'passage: {normalize_arabic(c["text"])}' for c in chunks]
    embeddings = embed_model.encode(
        texts_to_embed,
        batch_size=16,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        chunk_id = f"{contract['hash']}_{i}"
        all_ids.append(chunk_id)
        all_docs.append(chunk['text'])
        all_metas.append({
            'file': contract['file'],
            'contract_type': contract['type'],
            'article': chunk['article'],
            'chunk_index': i
        })
        all_embeds.append(emb.tolist())

# Insérer par batch
BATCH_SIZE = 100
for i in range(0, len(all_ids), BATCH_SIZE):
    collection.add(
        ids=all_ids[i:i+BATCH_SIZE],
        documents=all_docs[i:i+BATCH_SIZE],
        metadatas=all_metas[i:i+BATCH_SIZE],
        embeddings=all_embeds[i:i+BATCH_SIZE]
    )

print(f'\n✅ ChromaDB prêt: {collection.count()} chunks indexés')
print(f'   Types de contrats: {set(m["contract_type"] for m in all_metas)}')

🔄 Chargement du modèle: intfloat/multilingual-e5-large
   (Premier lancement: téléchargement ~560MB)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Modèle chargé

🔢 Génération des embeddings...


Contrats:   0%|          | 0/42 [00:00<?, ?it/s]


✅ ChromaDB prêt: 148 chunks indexés
   Types de contrats: {'عقد_قرض', 'عقد_بيع', 'عقد_وكالة', 'عقد_عمل', 'عقد_إيجار', 'عقد_شراكة', 'عقد_عام'}


---
## 🤖 Étape 5 — Chargement du LLM (Gemma-2-9B)

In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── Choix du modèle ───────────────────────────────────────────────────────────
# Option 1: google/gemma-2-9b-it     — Excellent arabe, 9B params (GPU T4 OK avec 4-bit)
# Option 2: Qwen/Qwen2.5-7B-Instruct — Très bon arabe, 7B params
# Option 3: CohereForAI/aya-expanse-8b — Spécialisé multilingue arabe
# → Kaggle T4 GPU: 15GB VRAM — Qwen2.5-7B est le plus fiable

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # Changez ici si besoin

print(f'🤖 Chargement: {MODEL_ID}')
print(f'   GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Quantisation 4-bit pour rentrer dans le GPU Kaggle T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.eval()

print(f'\n✅ Modèle chargé en mémoire')
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   VRAM utilisée: {used:.1f}/{total:.1f} GB')

🤖 Chargement: Qwen/Qwen2.5-7B-Instruct
   GPU disponible: True
   VRAM: 15.6 GB


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


✅ Modèle chargé en mémoire
   VRAM utilisée: 4.1/15.6 GB


---
## 🔍 Étape 6 — Moteur RAG

In [26]:
# ── Informations sur les types de contrats marocains ─────────────────────────
MOROCCAN_CONTRACT_INFO = {
    'عقد_إيجار': {
        'title': 'عقد كراء سكني',
        'law': 'القانون رقم 67.12 المتعلق بتنظيم العلاقات بين المكري والمكتري',
        'parties': ['المكري (الطرف الأول)', 'المكتري (الطرف الثاني)'],
        'clauses': ['وصف العقار', 'مدة الكراء', 'مبلغ الكراء وطريقة الأداء',
                    'الضمان', 'التزامات المكري', 'التزامات المكتري', 'فسخ العقد']
    },
    'عقد_بيع': {
        'title': 'عقد البيع',
        'law': 'ظهير الالتزامات والعقود — الفصول 478 إلى 618',
        'parties': ['البائع (الطرف الأول)', 'المشتري (الطرف الثاني)'],
        'clauses': ['وصف المبيع', 'الثمن وطريقة الأداء', 'نقل الملكية', 'ضمان الاستحقاق']
    },
    'عقد_عمل': {
        'title': 'عقد الشغل',
        'law': 'مدونة الشغل المغربية — القانون رقم 65.99',
        'parties': ['المشغل (الطرف الأول)', 'الأجير (الطرف الثاني)'],
        'clauses': ['طبيعة العمل', 'الأجر والامتيازات', 'مدة العقد',
                    'فترة التجربة', 'أوقات العمل', 'الإجازات', 'الإنهاء']
    },
    'عقد_شراكة': {
        'title': 'عقد الشركة',
        'law': 'القانون رقم 5.96 المتعلق بشركات الأشخاص',
        'parties': ['الشريك الأول', 'الشريك الثاني'],
        'clauses': ['موضوع الشركة', 'رأس المال وتوزيع الحصص',
                    'توزيع الأرباح', 'تسيير الشركة', 'حل الشركة']
    },
    'عقد_مقاولة': {
        'title': 'عقد المقاولة',
        'law': 'ظهير الالتزامات والعقود — الفصول 723 إلى 769',
        'parties': ['صاحب المشروع (الطرف الأول)', 'المقاول (الطرف الثاني)'],
        'clauses': ['وصف الأشغال', 'الأثمان وطريقة الأداء', 'المدة', 'الضمانات']
    }
}

# ── Fonction de retrieval ─────────────────────────────────────────────────────
def retrieve(query: str, contract_type: str = None, n: int = 5) -> list:
    """Récupère les chunks les plus pertinents depuis ChromaDB."""
    q_emb = embed_model.encode(
        f'query: {normalize_arabic(query)}',
        normalize_embeddings=True
    ).tolist()
    where = {'contract_type': contract_type} if contract_type else None
    n_real = min(n, collection.count())
    if n_real == 0:
        return []
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=n_real,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )
    return [
        {'text': doc, 'meta': meta, 'score': round(1 - dist, 3)}
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )
    ]

# ── Génération avec le LLM ────────────────────────────────────────────────────
SYSTEM_PROMPT = """أنت محامٍ مغربي متخصص في صياغة العقود القانونية الرسمية.
تصيغ عقوداً قانونية مغربية احترافية ومتوافقة مع التشريع المغربي النافذ.

قواعد الصياغة:
- ابدأ دائماً بـ "بسم الله الرحمن الرحيم" ثم "المملكة المغربية"
- استخدم المصطلحات القانونية المغربية (مكري/مكتري، مشغل/أجير...)
- أشر إلى القانون المغربي المنطبق في الديباجة
- رقّم البنود بالأرقام العربية (الأول، الثاني...)
- اختم بخانة التوقيع: الطرف الأول والطرف الثاني والتاريخ والمكان
- لا تضف أي تعليق أو شرح خارج نص العقد"""

def generate_contract(prompt: str, max_new_tokens: int = 2048) -> str:
    """Génère du texte avec le LLM chargé."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def build_rag_prompt(request: str, contract_type: str, info: dict, context_chunks: list) -> str:
    """Construit le prompt RAG complet."""
    ctx = ''
    if context_chunks:
        ctx = '\n## أمثلة من عقود مغربية مرجعية:\n'
        for i, c in enumerate(context_chunks[:3]):
            ctx += f'\n### مثال {i+1} — {c["meta"].get("article", "")}\n'
            ctx += c['text'][:500] + '...\n'
    clauses_list = '\n'.join(f'- {cl}' for cl in info.get('clauses', []))
    return f"""## نوع العقد: {info.get('title', contract_type)}
## القانون المنطبق: {info.get('law', 'القانون المدني المغربي')}
## البنود المطلوبة:
{clauses_list}
{ctx}
## الطلب:
{request}

اكتب العقد القانوني المغربي الكامل:"""

print('✅ Moteur RAG configuré et prêt')

✅ Moteur RAG configuré et prêt


---
## 🎯 Étape 7 — Génération de contrats (Démo Managers)
### Exemple 1 : Contrat de bail (Kira) — Casablanca

In [27]:
from IPython.display import display, Markdown

# ── Paramètres du contrat ─────────────────────────────────────────────────────
CONTRACT_TYPE = 'عقد_إيجار'
info = MOROCCAN_CONTRACT_INFO[CONTRACT_TYPE]

USER_REQUEST = """
قم بصياغة عقد كراء سكني بالمواصفات التالية:
- المكري: السيد أحمد بنعلي، المقيم بالدار البيضاء، عين الذئاب، ويحمل ب.و رقم AB123456
- المكتري: السيد يوسف الأمراني، المقيم بالرباط، ويحمل ب.و رقم CD789012
- العقار: شقة من 3 غرف، الطابق الثاني، عمارة الفتح، شارع محمد الخامس، الدار البيضاء
- مبلغ الكراء: 4500 درهم شهرياً
- مدة الكراء: سنة واحدة قابلة للتجديد، تبدأ من 1 يناير 2025
- الضمان: شهران (9000 درهم)
- المدينة: الدار البيضاء
"""

# ── RAG: récupérer les exemples pertinents ────────────────────────────────────
print('🔍 Recherche de contexte RAG...')
context = retrieve(USER_REQUEST, CONTRACT_TYPE, n=5)
print(f'   {len(context)} chunks récupérés')
if context:
    print(f'   Meilleur score de similarité: {context[0]["score"]}')

# ── Construire le prompt et générer ──────────────────────────────────────────
print('\n⚡ Génération du contrat...')
prompt = build_rag_prompt(USER_REQUEST, CONTRACT_TYPE, info, context)

contract_text = generate_contract(prompt, max_new_tokens=2500)

print('\n' + '='*70)
print('CONTRAT GÉNÉRÉ:')
print('='*70)
display(Markdown(f'```\n{contract_text}\n```'))

🔍 Recherche de contexte RAG...
   5 chunks récupérés
   Meilleur score de similarité: 0.874

⚡ Génération du contrat...

CONTRAT GÉNÉRÉ:


```
بسم الله الرحمن الرحيم  
المملكة المغربية  

عـقـد كـراء

بين الموقعين اسفله:
الطرف الأول: السيد أحمد بنعلي، مغربي الجنسية، مزداد بتاريخ 01/01/1975، الحامل لبطاقة التعريف الوطنية رقم AB123456، القاطن بالدار البيضاء، عين الذئاب.
            المكري من جهة أولى

والطرف الثاني: السيد يوسف الأمراني، مغربي الجنسية، مزداد بتاريخ 15/03/1980، الحامل لبطاقة التعريف الوطنية رقم CD789012، القاطن بالرباط.
            المكتري من جهة ثانية

وبعد أن وافق الطرفان على ما يلي:

الأول: وصف العقار:
- الشقة الكائنة بالطابق الثاني من العمارة المعروفة باسم "عمارة الفتح"، شارع محمد الخامس، الدار البيضاء.

الثاني: مدة الكراء:
- مدة الكراء تبتدئ من 01/01/2025 وتستمر لمدة سنة واحدة قابلة للتجديد برضى الطرفين.

الثالث: مبلغ الكراء وطريقة الأداء:
- مبلغ الكراء قدره 4500 درهم شهرياً، ويؤدى كل شهر قبل الخامس من الشهر التالي له.

الرابع: الضمان:
- الضمان قدره 9000 درهم (تسعة آلاف درهم)، يتم دفعه بمجرد توقيع العقد.

الخامس: التزامات المكري:
- توفير الشقة في حالة جيدة للاستخدام السكني.
- القيام بأعمال الصيانة اللازمة للشقة.

السادس: التزامات المكتري:
- استخدام الشقة لأغراض سكنية فقط.
- الإلتزام بدفع الكراء في الوقت المحدد.
- إفراغ الشقة في حالة إنهاء الكراء أو فسخ العقد.

السابع: فسخ العقد:
- يمكن فسخ العقد إذا لم يتم دفع الكراء في المواعيد المحددة.
- في حالة عدم إفراغ الشقة في الوقت المحدد، يتعين دفع مبلغ إضافي قدره 2000 درهم (ألفي درهم) عن كل شهر تأخير.

وحيث أنه تم الاتفاق على جميع النقاط الواردة أعلاه، فإن الطرفين يوقعون على هذا العقد في الدار البيضاء، في تاريخ 01/01/2025.

__________________________           ________________________
المكري: السيد أحمد بنعلي                    المكتري: السيد يوسف الأمراني

المدينة: الدار البيضاء، التاريخ: 01/01/2025
```

### Exemple 2 : Contrat de travail (CDI)

In [30]:
CONTRACT_TYPE_2 = 'عقد_عمل'
info_2 = MOROCCAN_CONTRACT_INFO[CONTRACT_TYPE_2]

USER_REQUEST_2 = """
قم بصياغة عقد شغل غير محدد المدة (CDI) بالمواصفات التالية:
- المشغل: شركة "تكنوليدر" للاستشارات، ذ.م.م، مقرها عين السبع، الدار البيضاء
- الأجير: السيدة فاطمة الزهراء خالد، مهندسة في المعلوميات
- المنصب: مهندسة تطوير برمجيات أول
- الأجر: 12000 درهم صافية شهرياً
- فترة التجربة: 3 أشهر قابلة للتجديد مرة واحدة
- أوقات العمل: 44 ساعة أسبوعياً
- تاريخ بداية العقد: 15 فبراير 2025
- المزايا الإضافية: تأمين صحي، تعويض نقل 500 درهم/شهر
"""

print('🔍 RAG retrieval...')
context_2 = retrieve(USER_REQUEST_2, CONTRACT_TYPE_2, n=5)
print(f'   {len(context_2)} chunks récupérés')

print('⚡ Génération...')
prompt_2 = build_rag_prompt(USER_REQUEST_2, CONTRACT_TYPE_2, info_2, context_2)
contract_text_2 = generate_contract(prompt_2, max_new_tokens=2500)

print('\n' + '='*70)
display(Markdown(f'```\n{contract_text_2}\n```'))

🔍 RAG retrieval...
   5 chunks récupérés
⚡ Génération...



```
بسم الله الرحمن الرحيم  
المملكة المغربية

**عقد الشغل**

المشغل: شركة "تكنوليدر" للاستشارات، ذ.م.م، مقرها عين السبع، الدار البيضاء (فيما بعد "المشغل")

والأجير: السيدة فاطمة الزهراء خالد، مهندسة في المعلوميات (فيما بعد "الأجير")

وبناء على مقتضيات مدونة الشغل المغربية — القانون رقم 65.99، فإن المشغل والأجير قد اتفقا على ما يلي:

**البند الأول**: طبيعة العمل
1. يكون الأجير مسؤولاً عن تنفيذ الأعمال المرتبطة بموقعه الوظيفي كمهندس تطوير برمجيات أول في شركة "تكنوليدر" للاستشارات، ذ.م.م.

**البند الثاني**: الأجر والامتيازات
1. يتم سداد الأجير للأجر الشهري بمبلغ 12,000 درهم صافيًا.
2. يتضمن الأجر المذكور أعلاه جميع التعويضات والمزايا الإضافية المحددة في هذا العقد.

**البند الثالث**: مدة العقد
1. يعتبر هذا العقد غير محدد المدة، ويستمر حتى إشعار آخر.

**البند الرابع**: فترة التجربة
1. يُمنح الأجير فترة تجريبية مدتها ثلاثة أشهر قابلة للتجديد مرة واحدة بناءً على موافقة المشغل.

**البند الخامس**: أوقات العمل
1. يتوجب على الأجير أن يعمل 44 ساعة أسبوعياً، وذلك وفقاً لجدول العمل المتفق عليه بين الجانبين.

**البند السادس**: الإجازات
1. يستفيد الأجير من الإجازات السنوية المحددة في مدونة الشغل المغربية.

**البند السابع**: الإنهاء
1. يمكن إنهاء هذا العقد من قبل المشغل لأسباب مبررة وفقاً للمادة 78 من مدونة الشغل المغربية.
2. في حال إنهاء العقد من قبل الأجير، يجب أن يقدم استقالته مسبقاً بمدة شهر واحد.

**البند الثامن**: المزايا الإضافية
1. يوفر المشغل تأمين صحي للأجير.
2. يتعهد المشغل بتقديم تعويض نقل قدره 500 درهم شهرياً.

**الخانة الأولى للتواقيع**
_________________________ المشغل
(توقيع)
_________________________ الأجير
(توقيع)

دار البيضاء في 15 فبراير 2025
```

---
## 📊 Étape 8 — Export DOCX de qualité (pour présentation)

In [31]:
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from datetime import datetime
import io

def set_rtl(paragraph):
    """Configure un paragraphe en RTL pour l'arabe."""
    pPr = paragraph._p.get_or_add_pPr()
    bidi = OxmlElement('w:bidi')
    pPr.append(bidi)
    jc = OxmlElement('w:jc')
    jc.set(qn('w:val'), 'right')
    pPr.append(jc)

def set_arabic_font(run, size=12, bold=False, color=None):
    """Applique la police arabe Times New Roman (Arabic)."""
    run.font.name = 'Times New Roman'
    run.font.size = Pt(size)
    run.font.bold = bold
    if color:
        run.font.color.rgb = RGBColor(*color)
    # Force Arabic font
    rPr = run._r.get_or_add_rPr()
    rFonts = OxmlElement('w:rFonts')
    rFonts.set(qn('w:hint'), 'eastAsia')
    rFonts.set(qn('w:ascii'), 'Times New Roman')
    rFonts.set(qn('w:hAnsi'), 'Times New Roman')
    rFonts.set(qn('w:cs'), 'Traditional Arabic')
    rPr.insert(0, rFonts)

def add_header_line(doc, color=(192, 0, 0)):
    """Ajoute une ligne décorative de couleur."""
    p = doc.add_paragraph()
    pPr = p._p.get_or_add_pPr()
    pBdr = OxmlElement('w:pBdr')
    bottom = OxmlElement('w:bottom')
    bottom.set(qn('w:val'), 'single')
    bottom.set(qn('w:sz'), '6')
    bottom.set(qn('w:space'), '1')
    bottom.set(qn('w:color'), '%02X%02X%02X' % color)
    pBdr.append(bottom)
    pPr.append(pBdr)
    return p

def contract_to_docx(contract_text: str, title: str, output_path: str):
    """Génère un DOCX de qualité professionnelle en arabe."""
    doc = Document()

    # ── Marges ────────────────────────────────────────────────────────────────
    section = doc.sections[0]
    section.top_margin = Cm(2.5)
    section.bottom_margin = Cm(2.5)
    section.left_margin = Cm(3)
    section.right_margin = Cm(3)

    # ── En-tête: Royaume du Maroc ─────────────────────────────────────────────
    header_p = doc.add_paragraph()
    set_rtl(header_p)
    header_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = header_p.add_run('المملكة المغربية')
    set_arabic_font(run, size=11, bold=True, color=(128, 0, 0))

    bismillah_p = doc.add_paragraph()
    set_rtl(bismillah_p)
    bismillah_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = bismillah_p.add_run('بسم الله الرحمن الرحيم')
    set_arabic_font(run, size=14, bold=True)

    add_header_line(doc)

    # ── Titre du contrat ──────────────────────────────────────────────────────
    title_p = doc.add_paragraph()
    set_rtl(title_p)
    title_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title_p.add_run(title)
    set_arabic_font(run, size=18, bold=True, color=(0, 70, 127))
    title_p.space_after = Pt(12)

    add_header_line(doc)
    doc.add_paragraph()  # espacement

    # ── Corps du contrat ──────────────────────────────────────────────────────
    lines = contract_text.split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            doc.add_paragraph()
            continue

        p = doc.add_paragraph()
        set_rtl(p)

        # Détecter les titres de bandes
        is_article = bool(re.match(
            r'^(البند|المادة|الفصل|أولاً|ثانياً|ثالثاً)',
            line
        ))
        is_header = any(kw in line for kw in ['بسم الله', 'المملكة', 'عقد '])

        run = p.add_run(line)
        if is_article:
            set_arabic_font(run, size=13, bold=True, color=(0, 70, 127))
            p.space_before = Pt(8)
        elif is_header:
            set_arabic_font(run, size=12, bold=True)
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        else:
            set_arabic_font(run, size=11)
        p.paragraph_format.line_spacing = Pt(18)

    # ── Zone de signature ─────────────────────────────────────────────────────
    doc.add_paragraph()
    add_header_line(doc)

    sig_table = doc.add_table(rows=3, cols=2)
    sig_table.style = 'Table Grid'

    headers = ['الطرف الأول', 'الطرف الثاني']
    for i, cell in enumerate(sig_table.rows[0].cells):
        p = cell.paragraphs[0]
        set_rtl(p)
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = p.add_run(headers[i])
        set_arabic_font(run, size=12, bold=True, color=(0, 70, 127))

    for cell in sig_table.rows[1].cells:
        p = cell.paragraphs[0]
        set_rtl(p)
        run = p.add_run('الاسم: ____________________')
        set_arabic_font(run, size=11)

    for cell in sig_table.rows[2].cells:
        p = cell.paragraphs[0]
        set_rtl(p)
        run = p.add_run('\nالتوقيع:\n\n\n')
        set_arabic_font(run, size=11)

    # ── Pied de page ──────────────────────────────────────────────────────────
    footer = doc.sections[0].footer
    fp = footer.paragraphs[0]
    fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = fp.add_run(f'تم التحرير بتاريخ: {datetime.now().strftime("%d/%m/%Y")}  —  نموذج مولّد بالذكاء الاصطناعي')
    set_arabic_font(run, size=9, color=(128, 128, 128))

    doc.save(output_path)
    print(f'✅ DOCX sauvegardé: {output_path}')


# ── Exporter les deux contrats générés ───────────────────────────────────────
contract_to_docx(
    contract_text,
    title='عقد كراء سكني',
    output_path='/kaggle/working/عقد_كراء_سكني.docx'
)

contract_to_docx(
    contract_text_2,
    title='عقد الشغل',
    output_path='/kaggle/working/عقد_الشغل.docx'
)

print('\n📁 Fichiers disponibles dans /kaggle/working/')

✅ DOCX sauvegardé: /kaggle/working/عقد_كراء_سكني.docx
✅ DOCX sauvegardé: /kaggle/working/عقد_الشغل.docx

📁 Fichiers disponibles dans /kaggle/working/


---
## 📈 Étape 9 — Métriques de qualité (pour les managers)

In [32]:
import time
from IPython.display import display, HTML

def evaluate_contract_quality(contract_text: str, contract_type: str) -> dict:
    """Évalue la qualité d'un contrat généré."""
    info = MOROCCAN_CONTRACT_INFO.get(contract_type, {})
    required_clauses = info.get('clauses', [])
    
    metrics = {}
    
    # 1. Présence des éléments obligatoires
    metrics['bismillah'] = 'بسم الله' in contract_text
    metrics['maroc_header'] = 'المملكة المغربية' in contract_text
    metrics['has_parties'] = 'الطرف الأول' in contract_text and 'الطرف الثاني' in contract_text
    metrics['has_signature'] = any(kw in contract_text for kw in ['التوقيع', 'توقيع', 'إمضاء'])
    metrics['has_date'] = any(kw in contract_text for kw in ['التاريخ', 'بتاريخ', '2025', '2024'])
    
    # 2. Couverture des bandes requises
    clause_found = []
    for clause in required_clauses:
        # Chercher les mots-clés de chaque bande
        keywords = clause.split()
        found = any(kw in contract_text for kw in keywords if len(kw) > 3)
        clause_found.append(found)
    
    metrics['clause_coverage'] = sum(clause_found) / len(clause_found) if clause_found else 0
    metrics['clauses_found'] = sum(clause_found)
    metrics['clauses_total'] = len(clause_found)
    
    # 3. Longueur et richesse
    metrics['char_count'] = len(contract_text)
    metrics['word_count'] = len(contract_text.split())
    metrics['arabic_ratio'] = sum(1 for c in contract_text if '\u0600' <= c <= '\u06FF') / max(len(contract_text), 1)
    
    # 4. Score global
    binary_checks = [v for k, v in metrics.items() if isinstance(v, bool)]
    score = (sum(binary_checks) / len(binary_checks) * 0.5 + metrics['clause_coverage'] * 0.5)
    metrics['overall_score'] = round(score * 100, 1)
    
    return metrics

# ── Évaluer les contrats générés ──────────────────────────────────────────────
m1 = evaluate_contract_quality(contract_text, 'عقد_إيجار')
m2 = evaluate_contract_quality(contract_text_2, 'عقد_عمل')

html = """
<style>
  .metrics-table { border-collapse: collapse; width: 100%; font-family: Arial; }
  .metrics-table th { background: #1a3a5c; color: white; padding: 10px; text-align: center; }
  .metrics-table td { padding: 8px 12px; border: 1px solid #ddd; text-align: center; }
  .metrics-table tr:nth-child(even) { background: #f5f8fc; }
  .score-high { color: #22aa44; font-weight: bold; font-size: 1.2em; }
  .score-med  { color: #f09020; font-weight: bold; }
  .score-low  { color: #cc2222; font-weight: bold; }
  .check-yes  { color: #22aa44; font-size: 1.3em; }
  .check-no   { color: #cc2222; font-size: 1.3em; }
</style>
<h3>📊 Évaluation de la qualité des contrats générés</h3>
<table class='metrics-table'>
  <tr>
    <th>Critère</th>
    <th>Contrat de Bail</th>
    <th>Contrat de Travail</th>
  </tr>
"""

rows = [
    ('✅ Bismillah présent',        'bismillah',        'bool'),
    ('✅ En-tête Maroc',            'maroc_header',     'bool'),
    ('✅ Parties mentionnées',       'has_parties',      'bool'),
    ('✅ Zone signature',           'has_signature',    'bool'),
    ('✅ Date mentionnée',          'has_date',         'bool'),
    ('📋 Bandes couvertes',         'clause_coverage',  'pct'),
    ('📝 Nombre de mots',           'word_count',       'num'),
    ('🔤 Ratio texte arabe',        'arabic_ratio',     'pct'),
    ('⭐ Score global',             'overall_score',    'score'),
]

for label, key, fmt in rows:
    v1, v2 = m1.get(key), m2.get(key)
    def fmt_val(v, f):
        if f == 'bool':
            return f"<span class='check-{'yes' if v else 'no'}'>{'✓' if v else '✗'}</span>"
        elif f == 'pct':
            pct = v * 100 if v <= 1 else v
            cls = 'score-high' if pct >= 70 else ('score-med' if pct >= 40 else 'score-low')
            return f"<span class='{cls}'>{pct:.0f}%</span>"
        elif f == 'score':
            cls = 'score-high' if v >= 70 else ('score-med' if v >= 50 else 'score-low')
            return f"<span class='{cls}'>{v}%</span>"
        else:
            return str(v)
    html += f"<tr><td>{label}</td><td>{fmt_val(v1, fmt)}</td><td>{fmt_val(v2, fmt)}</td></tr>"

html += '</table>'
display(HTML(html))

Critère,Contrat de Bail,Contrat de Travail
✅ Bismillah présent,✓,✓
✅ En-tête Maroc,✓,✓
✅ Parties mentionnées,✓,✗
✅ Zone signature,✓,✓
✅ Date mentionnée,✓,✓
📋 Bandes couvertes,100%,100%
📝 Nombre de mots,253,241
🔤 Ratio texte arabe,69%,72%
⭐ Score global,100.0%,90.0%


---
## 💬 Étape 10 — Interface chatbot interactive

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── UI ────────────────────────────────────────────────────────────────────────
style = {'description_width': 'initial'}

contract_type_dropdown = widgets.Dropdown(
    options=[
        ('عقد كراء سكني', 'عقد_إيجار'),
        ('عقد الشغل', 'عقد_عمل'),
        ('عقد البيع', 'عقد_بيع'),
        ('عقد الشراكة', 'عقد_شراكة'),
        ('عقد المقاولة', 'عقد_مقاولة'),
    ],
    value='عقد_إيجار',
    description='نوع العقد:',
    style=style, layout=widgets.Layout(width='350px')
)

user_input = widgets.Textarea(
    placeholder='اكتب متطلبات العقد هنا... (باللغة العربية)',
    description='الطلب:',
    style=style,
    layout=widgets.Layout(width='700px', height='120px')
)

generate_btn = widgets.Button(
    description='⚡ توليد العقد',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
export_btn = widgets.Button(
    description='📥 تحميل DOCX',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = widgets.Output()
last_contract = {'text': '', 'type': ''}

def on_generate(btn):
    with output_area:
        clear_output()
        if not user_input.value.strip():
            print('⚠️ يرجى إدخال متطلبات العقد')
            return
        
        ctype = contract_type_dropdown.value
        info = MOROCCAN_CONTRACT_INFO.get(ctype, {})
        
        print(f'🔍 استرجاع السياق من {collection.count()} نصوص قانونية...')
        ctx = retrieve(user_input.value, ctype, n=5)
        print(f'   {len(ctx)} مقاطع ذات صلة وجدت')
        print('⚡ جاري توليد العقد...')
        
        t0 = time.time()
        prompt = build_rag_prompt(user_input.value, ctype, info, ctx)
        result = generate_contract(prompt, max_new_tokens=2000)
        elapsed = time.time() - t0
        
        last_contract['text'] = result
        last_contract['type'] = ctype
        
        print(f'\n✅ تم التوليد في {elapsed:.1f} ثانية\n')
        print('=' * 60)
        print(result)
        print('=' * 60)

def on_export(btn):
    if not last_contract['text']:
        with output_area:
            print('⚠️ قم بتوليد عقد أولاً')
        return
    
    info = MOROCCAN_CONTRACT_INFO.get(last_contract['type'], {})
    title = info.get('title', 'عقد')
    filename = f'/kaggle/working/{title}_{datetime.now().strftime("%Y%m%d_%H%M")}.docx'
    contract_to_docx(last_contract['text'], title, filename)
    with output_area:
        print(f'\n📥 محفوظ: {filename}')

generate_btn.on_click(on_generate)
export_btn.on_click(on_export)

display(
    widgets.VBox([
        widgets.HTML('<h3 style="color:#1a3a5c">🇲🇦 نظام توليد العقود القانونية المغربية</h3>'),
        contract_type_dropdown,
        user_input,
        widgets.HBox([generate_btn, export_btn]),
        output_area
    ])
)

---
## 🗂️ Résumé de l'architecture pour les managers

| Composant | Technologie | Rôle |
|-----------|-------------|------|
| **Parsing PDF** | PyMuPDF | Extraction du texte arabe RTL |
| **Embeddings** | `multilingual-e5-large` | Vecteurs sémantiques arabes |
| **Base vectorielle** | ChromaDB | Recherche par similarité |
| **LLM** | Qwen2.5-7B / Gemma-2-9B | Génération en arabe |
| **Export** | python-docx | Documents Word professionnels |

### Points forts
- ✅ 100% local — aucune donnée ne sort de l'infrastructure
- ✅ Conforme aux lois marocaines (67.12, 65.99, Z.O.C.)
- ✅ Extension facile : ajouter des PDFs → re-ingérer → qualité améliorée
- ✅ Export DOCX signable directement